## 1. Install Dependencies

In [1]:
# tested on python 3.12.10
import zipfile

!curl -L https://github.com/mpj1234/ncnn-yolo26-android/releases/download/asserts/ultralytics-8.4.6.zip -o ultralytics.zip
with zipfile.ZipFile("ultralytics.zip", "r") as z:
    z.extractall("ultralytics-src")
%pip install -U torch torchvision --index-url https://download.pytorch.org/whl/cu128
%pip install -U ncnn==1.0.20260114 pnnx==20260112 roboflow
%pip install -e ./ultralytics-src


  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
  0      0   0      0   0      0      0      0                              0
  0      0   0      0   0      0      0      0                              0

  0      0   0      0   0      0      0      0                              0
100  2.59M 100  2.59M   0      0  3.07M      0                              0
100  2.59M 100  2.59M   0      0  3.07M      0                              0
100  2.59M 100  2.59M   0      0  3.07M      0                              0


Looking in indexes: https://download.pytorch.org/whl/cu128
  Using cached https://download.pytorch.org/whl/cu128/torch-2.10.0%2Bcu128-cp312-cp312-win_amd64.whl.metadata (29 kB)
  Using cached https://download.pytorch.org/whl/cu128/torchvision-0.25.0%2Bcu128-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached https://download.pytorch.org/whl/typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached https://download.pytorch.org/whl/jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2025.12.0-py3-none-any.whl.metadata (10 kB)
  Using cached https://download.pytorch.org/whl/setuptools-70.2.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached numpy-2.3.5-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached pillow-12.0.0-cp312-cp312-win_amd64.whl.metadata 


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached ncnn-1.0.20260114-cp312-cp312-win_amd64.whl.metadata (28 kB)
  Using cached pnnx-20260112-py3-none-win_amd64.whl.metadata (6.4 kB)
  Using cached roboflow-1.2.16-py3-none-any.whl.metadata (10 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached portalocker-3.2.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached idna-3.7-py3-none-any.whl.metadata (9.9 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.0-cp312-cp312-win_amd64.whl.metadata (5.2 kB)
  Using cached matplotlib-3.10.8-cp312-cp312-win_amd64.whl.metadata (52 kB)
  Using cached opencv_python_headless-4.10.0.84-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached pillow_avif_plugin-1.5.5-cp312-cp312-win_amd64.whl.metadata (2.3 kB


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Obtaining file:///C:/Users/thesp/Documents/Projects/Unibots/BARF/ultralytics-src
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached polars-1.39.0-py3-none-any.whl.metadata (10 kB)
  Using cached ultralytics_thop-2.0.18-py3-none-any.whl.metadata (14 kB)
  Using cached polars_runtime_32-1.39.0-cp310-abi3-win_amd64.whl.metadata (1.5 kB)
Using cached polars-1.39.0-py3-none-any.whl (823 kB)
Using cached polars_runtime_32-1.39.0-cp310-abi3-win_amd64.whl (47.0 MB)
Using c


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Download and prepare dataset

In [2]:
from roboflow import Roboflow
rf = Roboflow(api_key="Mt0m4dCKTTlNf29OV3zm")
project = rf.workspace("dylans-workspace-3init").project("my-first-project-sbk3a")
version = project.version(1)
dataset = version.download("yolo26") # This downloads the images AND the data.yaml

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to My-First-Project-1 in yolo26:: 100%|██████████| 262/262 [00:00<00:00, 1519.33it/s]


### 2.1 Patch dataset with missing labels

In [3]:
from pathlib import Path

dataset_root = Path("My-First-Project-1")
OLD_BLOCK = """nc: 1
names: ['sports_ball']"""
NEW_BLOCK = """names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: airplane
  5: bus
  6: train
  7: truck
  8: boat
  9: traffic light
  10: fire hydrant
  11: stop sign
  12: parking meter
  13: bench
  14: bird
  15: cat
  16: dog
  17: horse
  18: sheep
  19: cow
  20: elephant
  21: bear
  22: zebra
  23: giraffe
  24: backpack
  25: umbrella
  26: handbag
  27: tie
  28: suitcase
  29: frisbee
  30: skis
  31: snowboard
  32: sports ball
  33: kite
  34: baseball bat
  35: baseball glove
  36: skateboard
  37: surfboard
  38: tennis racket
  39: bottle
  40: wine glass
  41: cup
  42: fork
  43: knife
  44: spoon
  45: bowl
  46: banana
  47: apple
  48: sandwich
  49: orange
  50: broccoli
  51: carrot
  52: hot dog
  53: pizza
  54: donut
  55: cake
  56: chair
  57: couch
  58: potted plant
  59: bed
  60: dining table
  61: toilet
  62: tv
  63: laptop
  64: mouse
  65: remote
  66: keyboard
  67: cell phone
  68: microwave
  69: oven
  70: toaster
  71: sink
  72: refrigerator
  73: book
  74: clock
  75: vase
  76: scissors
  77: teddy bear
  78: hair drier
  79: toothbrush"""
def rewrite_data_yaml( dry_run: bool = False):
    path = dataset_root / "data.yaml"
    text = path.read_text(encoding="utf-8")

    if OLD_BLOCK not in text:
        raise ValueError("Target block not found. Nothing replaced.")

    updated = text.replace(OLD_BLOCK, NEW_BLOCK, 1)

    if dry_run:
        print("Dry run: replacement would be applied.")
        return

    path.write_text(updated, encoding="utf-8")
    print(f"Updated: {path}")

def rewrite_label_file(path: Path, target_class: int, dry_run: bool = False):
    changed_lines = 0
    original = path.read_text(encoding="utf-8").splitlines(keepends=True)
    updated = []

    for line in original:
        stripped = line.strip()

        # Keep blank lines unchanged
        if not stripped:
            updated.append(line)
            continue

        # Split into: first token (class id) + rest (coords/polygon points)
        parts = stripped.split(maxsplit=1)
        first = parts[0]
        rest = parts[1] if len(parts) > 1 else ""

        # Only rewrite lines that start with a numeric class id
        try:
            float(first)
        except ValueError:
            updated.append(line)
            continue

        new_line = f"{target_class} {rest}".rstrip()
        if line.endswith("\n"):
            new_line += "\n"

        if new_line != line:
            changed_lines += 1
        updated.append(new_line)

    if changed_lines and not dry_run:
        path.write_text("".join(updated), encoding="utf-8")

    return changed_lines

def change_label_classes(dry_run: bool = True):
    target_class = 32
    label_dirs = [
        dataset_root / "train" / "labels",
        dataset_root / "valid" / "labels",
        dataset_root / "test" / "labels",
    ]

    total_files = 0
    touched_files = 0
    total_lines_changed = 0

    for label_dir in label_dirs:
        if not label_dir.exists():
            print(f"Skipping missing directory: {label_dir}")
            continue

        for txt_file in label_dir.rglob("*.txt"):
            total_files += 1
            changed = rewrite_label_file(txt_file, target_class, dry_run=dry_run)
            if changed:
                touched_files += 1
                total_lines_changed += changed
                print(f"Updated {txt_file} ({changed} line(s))")

    mode = "DRY RUN" if dry_run else "WRITE"
    print(f"\n[{mode}] Scanned files: {total_files}")
    print(f"[{mode}] Files changed: {touched_files}")
    print(f"[{mode}] Label lines changed: {total_lines_changed}")


dry_run = False  # Set to False to actually write changes
rewrite_data_yaml(dry_run=dry_run)
change_label_classes(dry_run=dry_run)

Updated: My-First-Project-1\data.yaml
Updated My-First-Project-1\train\labels\photo-1773138578280_jpg.rf.a408788a68e87aa74f80f80188677e71.txt (1 line(s))
Updated My-First-Project-1\train\labels\photo-1773138582198_jpg.rf.b6b67dedacf4c5eca861d89ee84a99bf.txt (1 line(s))
Updated My-First-Project-1\train\labels\photo-1773138585336_jpg.rf.564b09a948c8dd76f66e2a79b236b4e1.txt (1 line(s))
Updated My-First-Project-1\train\labels\photo-1773138588293_jpg.rf.d0064cc60c47f82dd8cc793d995feba2.txt (1 line(s))
Updated My-First-Project-1\train\labels\photo-1773138592333_jpg.rf.347a89f6f2f43d2faa3bc799e70d374b.txt (1 line(s))
Updated My-First-Project-1\train\labels\photo-1773138602119_jpg.rf.4bf832c44b250698bb8236e7f6e47017.txt (1 line(s))
Updated My-First-Project-1\train\labels\photo-1773138605935_jpg.rf.f018751aadde335b99764113bbd273e5.txt (1 line(s))
Updated My-First-Project-1\train\labels\photo-1773138608525_jpg.rf.cab563f165519fb027d5e6c31795a6a8.txt (1 line(s))
Updated My-First-Project-1\train\l

# 3. Train

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3070 Laptop GPU


In [2]:
from ultralytics import YOLO, settings
settings.reset() # reset the output dirs if different envs messed with it


# 1. Load a pretrained YOLO11n model
model = YOLO('yolo26n.pt')

# 2. Train the model
# 'data.yaml' contains paths to your orange ball images and class names
model.train(data="My-First-Project-1/data.yaml", epochs=100, imgsz=640,batch=32)

New https://pypi.org/project/ultralytics/8.4.22 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.6  Python-3.12.10 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=My-First-Project-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([32])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002050FDECA40>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048

## 3. Export YOLO26 NCNN

In [1]:
train_dir = "runs/detect/train/weights"

In [2]:
from pathlib import Path
from ultralytics import YOLO
YOLO(Path(train_dir)/"best.pt").export(**{
    'format': 'ncnn',
    'opset': 20,
    'simplify': True,
    'batch': 1,
    'imgsz': 640,
})

Ultralytics 8.4.6  Python-3.12.10 torch-2.10.0+cu128 CPU (AMD Ryzen 7 5800H with Radeon Graphics)
YOLO26n summary (fused): 122 layers, 2,408,932 parameters, 0 gradients, 5.4 GFLOPs

PyTorch: starting from 'runs\detect\train\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8400, 84) (5.3 MB)

NCNN: starting export with NCNN 1.0.20260114 and PNNX 20260112...
NCNN: export success  22.6s, saved as 'runs\detect\train\weights\best_ncnn_model' (4.7 MB)

Export complete (23.1s)
Results saved to C:\Users\thesp\Documents\Projects\Unibots\BARF\runs\detect\train\weights
Predict:         yolo predict task=detect model=runs\detect\train\weights\best_ncnn_model imgsz=640 
Validate:        yolo val task=detect model=runs\detect\train\weights\best_ncnn_model imgsz=640 data=My-First-Project-1/data.yaml  
Visualize:       https://netron.app


'runs\\detect\\train\\weights\\best_ncnn_model'

### 3.1 Copy the exported model to app assets

In [3]:
import shutil
from pathlib import Path

assets_dir = Path("app") / "src" / "main" / "assets"
renames = {
    "model.ncnn.bin": "yolo26n.ncnn.bin",
    "model.ncnn.param": "yolo26n.ncnn.param",
}
for src, dst in renames.items():
    shutil.copy((Path(train_dir)/"best_ncnn_model"/src).resolve(), (assets_dir / dst).resolve())
    print(f"Copied {src} -> {assets_dir / dst}")

Copied model.ncnn.bin -> app\src\main\assets\yolo26n.ncnn.bin
Copied model.ncnn.param -> app\src\main\assets\yolo26n.ncnn.param
